# Loading Pretrained GPT-2 Weights

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import tiktoken
from minigpt.pretrained import load_gpt2
from minigpt.generate import generate

/Users/dondapatisushanth.reddy/Documents/personal/projects/minigpt/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Pretrained Model

In [2]:
model = load_gpt2('gpt2')
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')
print(f'Weight tying: {model.tok_emb.weight is model.lm_head.weight}')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 49494.26it/s]


Parameters: 124,439,808
Weight tying: True


## 2. Generate Text

In [9]:
tokenizer = tiktoken.get_encoding('gpt2')

def gen(prompt, max_new_tokens=20):
    input_ids = torch.tensor([tokenizer.encode(prompt)])
    with torch.no_grad():
        output_ids = generate(model, input_ids, max_new_tokens=max_new_tokens, context_length=1024)
    return tokenizer.decode(output_ids[0].tolist())

In [10]:
prompts = [
    'To be or not to be',
    'The meaning of life is',
    'In the beginning, there was',
    'Despite the positive reviews, I found the movie',
    'Hello, my name is',
]

for prompt in prompts:
    print(f'--- {prompt} ---')
    print(gen(prompt))
    print()

--- To be or not to be ---
To be or not to be, the only thing that matters is that you're a good person.

I'm not saying

--- The meaning of life is ---
The meaning of life is not the same as the meaning of death.

The meaning of life is not the same as

--- In the beginning, there was ---
In the beginning, there was no way to know what was going to happen.

"I was just trying to get my

--- Despite the positive reviews, I found the movie ---
Despite the positive reviews, I found the movie to be a bit too dark and too dark for my tastes. I was also disappointed with the ending

--- Hello, my name is ---
Hello, my name is John. I'm a writer, and I'm a writer. I'm a writer. I'm



## 3. Verify Against HuggingFace

Proof that our from-scratch model produces identical outputs.

In [5]:
from transformers import GPT2LMHeadModel

hf_model = GPT2LMHeadModel.from_pretrained('gpt2', cache_dir='../data/hf_cache')
hf_model.eval()

test_input = torch.tensor([[15496, 11, 616, 1438, 318]])  # "Hello, my name is"

with torch.no_grad():
    our_logits = model(test_input)
    hf_logits = hf_model(test_input).logits

max_diff = (our_logits - hf_logits).abs().max().item()
print(f'Max logit difference: {max_diff:.2e}')
print(f'Match: {torch.allclose(our_logits, hf_logits, atol=1e-4)}')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10729.35it/s]


Max logit difference: 8.39e-05
Match: True
